In [55]:
# imports
import pandas as pd
import matplotlib.pyplot as plt

import os

In [56]:
# directory path
dir_path = r"..\14_stocks_analysis\00_data"
if not os.path.isdir(dir_path):
    raise FileNotFoundError(f'Directory does not exist: {dir_path}')
print(f'Using data directory: {dir_path}')

Using data directory: ..\14_stocks_analysis\00_data


In [57]:
# concatenate all tabular files into a single dataframe
files = [
    file for file in os.listdir(dir_path)
    if file.lower().endswith(('.csv', '.xlsx', '.xls'))
]
if not files:
    raise ValueError(f'No CSV/XLSX/XLS files found in: {dir_path}')

def load_table(file_name):
    file_path = os.path.join(dir_path, file_name)
    if file_name.lower().endswith('.csv'):
        return pd.read_csv(file_path)
    return pd.read_excel(file_path)

# file name have format like 2603, 2602..first two digits are year and last two are month
# when concatenating, we need to add a column with the date extracted from the file name
def extract_date(file_name):
    year = int(file_name[:2]) + 2000
    month = int(file_name[2:4])
    return pd.Timestamp(year=year, month=month, day=1)

df = pd.concat([load_table(file).assign(date=extract_date(file)) for file in sorted(files)], ignore_index=True)

# lowercase column names
df.columns = df.columns.str.lower()

df.head()

,unnamed: 0,symbol,stock,market cap,price,fair value (%),z-score,f-score,m-score,value generation,date
0,1.0,Lululemon Athletica Inc.,Lululemon Athletica Inc.,20.11B,168.18,116.31,7.43,8.0,-2.78,Robust,2025-11-01
1,NaN,LULU,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-11-01
2,2.0,"PayPal Holdings, Inc.","PayPal Holdings, Inc.",58.63B,60.57,100.92,1.96,7.0,-2.99,Robust,2025-11-01
3,NaN,PYPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-11-01
4,3.0,Adobe Inc.,Adobe Inc.,139.08B,324.19,82.13,8.92,7.0,-2.49,Robust,2025-11-01


In [58]:
# create a copy of symbol column named 'symbol_copy'
df['symbol_copy'] = df['symbol']

In [59]:
# the series in df.symbol_copy, should be shifted backwards by 1 row, to align with the stock name in the same row
df['symbol_copy'] = df['symbol_copy'].shift(-1)

In [60]:
# drop all rows where column 'unnamed: 0' is null
df = df.dropna(subset=['unnamed: 0'])

In [61]:
# drop columns 'unnamed: 0', 'symbol'
df = df.drop(columns=['unnamed: 0', 'symbol'])

# rename column 'symbol_copy' to 'symbol'
df = df.rename(columns={'symbol_copy': 'symbol'})

# columns order 'date', 'symbol', 'stock', 'market cap', 'price', 'fair value (%)', 'z-score', 'f-score', 'm-score', 'value generation'
df = df[['date', 'symbol', 'stock', 'market cap', 'price', 'fair value (%)', 'z-score', 'f-score', 'm-score', 'value generation']]

In [62]:
df.columns

Index(['date', 'symbol', 'stock', 'market cap', 'price', 'fair value (%)',
       'z-score', 'f-score', 'm-score', 'value generation'],
      dtype='object')

In [65]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 250 entries, 0 to 498
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              250 non-null    datetime64[us]
 1   symbol            250 non-null    object        
 2   stock             250 non-null    object        
 3   market cap        250 non-null    object        
 4   price             250 non-null    float64       
 5   fair value (%)    250 non-null    float64       
 6   z-score           250 non-null    float64       
 7   f-score           250 non-null    object        
 8   m-score           250 non-null    float64       
 9   value generation  250 non-null    object        
dtypes: datetime64[us](1), float64(4), object(5)
memory usage: 21.5+ KB


In [68]:
# create a copy of the dataframe
df_copy = df.copy()

# keep only date, symbol columns
df_copy = df_copy[['date', 'symbol']]

# group by symbol and count the number of occurrences of each symbol
symbol_counts = df_copy.groupby('symbol').size().reset_index(name='count')

# sort the dataframe by count in descending order
symbol_counts = symbol_counts.sort_values(by='count', ascending=False)

# display the top 10 symbols with the most occurrences
print(symbol_counts.head(10))

  symbol  count
0   ACGL      1
1    ACN      1
2   ADBE      1
3   ADSK      1
4   ALGN      1
5   ANET      1
6    AOS      1
7    AVY      1
8    BAC      1
9    BBY      1


In [ ]:
# drop duplicates in df.symbol, keep last occurrence
df = df.drop_duplicates(subset='symbol', keep='last')

# merge

df.head(10)

,date,symbol,stock,market cap,price,fair value (%),z-score,f-score,m-score,value generation
22,2025-11-01,RJF,"Raymond James Financial, Inc.",30.45B,152.99,30.16,1.05,9.0,-3.10,Robust
38,2025-11-01,BBY,"Best Buy Co., Inc.",16.21B,76.45,5.65,11.22,8.0,-3.28,Robust
60,2025-11-01,HES,Hess Corporation,45.97B,148.97,49.78,3.86,8.0,-3.07,Resilient
62,2025-11-01,ACN,Accenture plc,156.81B,251.85,38.65,5.86,6.0,-3.83,Robust
74,2025-11-01,WAB,Westinghouse Air Brake Technologies Corporation,34.14B,200.22,26.03,3.87,9.0,-2.64,Resilient
80,2025-11-01,TSCO,Tractor Supply Company,28.08B,52.92,10.39,5.43,6.0,-3.85,Robust
82,2025-11-01,GD,General Dynamics Corporation,91.48B,340.34,8.80,4.01,6.0,-3.58,Robust
84,2025-11-01,FOXA,Fox Corporation,30.02B,65.69,6.93,3.15,6.0,-3.61,Robust
86,2025-11-01,HSY,The Hershey Company,0.00,186.00,5.43,4.71,7.0,-2.32,Robust
90,2025-11-01,NOC,Northrop Grumman Corporation,81.09B,566.70,3.36,3.07,7.0,-3.08,Robust


In [64]:
df.sample(10)

,date,symbol,stock,market cap,price,fair value (%),z-score,f-score,m-score,value generation
70,2025-11-01,QCOM,QUALCOMM Incorporated,176.04B,163.30,13.50,6.78,6.0,-3.09,Robust
260,2026-01-01,LKQ,LKQ Corporation,8.83B,34.35,36.10,2.69,7.0,-2.74,Robust
226,2026-01-01,MSCI,MSCI Inc.,43.66B,565.90,26.63,4.88,8.0,-2.55,Robust
250,2026-01-01,TMO,Thermo Fisher Scientific Inc.,203.93B,542.36,10.50,4.34,8.0,-2.65,Robust
170,2025-12-01,MO,"Altria Group, Inc.",97.00B,57.60,26.18,4.53,7.0,-3.03,Robust
298,2026-01-01,IT,"Gartner, Inc.",10.95B,152.03,123.82,3.56,7.0,-3.18,Resilient
20,2025-11-01,UPS,"United Parcel Service, Inc.",80.37B,94.66,30.48,3.09,8.0,-3.55,Robust
204,2026-01-01,ZTS,Zoetis Inc.,56.13B,126.73,56.37,6.42,8.0,-2.76,Robust
50,2025-11-01,ODFL,"Old Dominion Freight Line, Inc.",28.32B,134.28,30.16,17.59,5.0,-5.37,Robust
6,2025-11-01,ZTS,Zoetis Inc.,54.06B,122.06,71.10,6.98,8.0,-2.76,Robust
